# Base de Datos Vectorial: Pinecone
## Motor de búsqueda semántica de ofertas laborales

**Proyecto Integrador 1 — Ingeniería de Sistemas**

### Objetivo del notebook

Reemplazar el índice FAISS en memoria (usado en `03_Embeddings_BusquedaVectorial.ipynb`) por una base de datos vectorial persistente: **Pinecone**, en su capa gratuita.

Decisiones tomadas para esta primera iteración:
- **Qué se indexa**: solo las **3,760 plantillas únicas** de texto (Opción A) — igual que con FAISS, no cada una de las 1.6M ofertas individuales.
- **Por qué Pinecone primero**: es la opción más simple de poner en marcha (servicio administrado, sin instalar/mantener infraestructura), útil para validar rápido cómo se siente trabajar con una base de datos vectorial real antes de decidir si migrar a algo self-hosted (Qdrant/Chroma) para el despliegue final en el VPS.

Este notebook asume que ya ejecutaste `02_Preprocesamiento.ipynb` y `03_Embeddings_BusquedaVectorial.ipynb` al menos una vez, de forma que en Google Drive (`proyecto_integrador/data/processed/`) existan:
- `job_descriptions_clean.parquet` (dataset limpio, sin `template_id`)
- `job_id_template_map.parquet` (mapeo `Job Id -> template_id`)
- `plantillas_meta.parquet` (una fila por plantilla única, con su `texto_combinado`)
- `embeddings_plantillas.npy` (embeddings ya generados de esas plantillas)

## 1. Preparación e instalación

In [1]:
!pip install -q pinecone sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 15.3 MB/s eta 0:00:00


In [2]:
import time

import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)


## 2. Cargar plantillas, embeddings y dataset limpio desde Drive

No hace falta volver a generar embeddings — ya se calcularon una vez en el notebook 03 y se guardaron en Drive. Aquí solo se leen.

In [3]:
from google.colab import drive

drive.mount("/content/drive")
BASE_DIR = Path("/content/drive/MyDrive/proyecto_integrador")
RUTA_PROCESSED = BASE_DIR / "data" / "processed"

archivos_requeridos = [
    "job_descriptions_clean.parquet",
    "job_id_template_map.parquet",
    "plantillas_meta.parquet",
    "embeddings_plantillas.npy",
]
faltantes = [f for f in archivos_requeridos if not (RUTA_PROCESSED / f).exists()]
if faltantes:
    raise FileNotFoundError(
        f"Faltan estos archivos en {RUTA_PROCESSED}: {faltantes}. "
        "Ejecuta primero 02_Preprocesamiento.ipynb y 03_Embeddings_BusquedaVectorial.ipynb."
    )

print("Todos los archivos requeridos están disponibles.")


Mounted at /content/drive
Todos los archivos requeridos están disponibles.


In [4]:
plantillas = pd.read_parquet(RUTA_PROCESSED / "plantillas_meta.parquet")
embeddings = np.load(RUTA_PROCESSED / "embeddings_plantillas.npy")

df_clean = pd.read_parquet(RUTA_PROCESSED / "job_descriptions_clean.parquet")
mapeo_template = pd.read_parquet(RUTA_PROCESSED / "job_id_template_map.parquet")
df_clean = df_clean.merge(mapeo_template, on="Job Id", how="left")

print(f"Plantillas: {len(plantillas):,}  |  Embeddings: {embeddings.shape}")
print(f"Ofertas completas (con template_id reconstruido): {len(df_clean):,}")


Plantillas: 3,760  |  Embeddings: (3760, 384)
Ofertas completas (con template_id reconstruido): 1,615,940


In [5]:
from google.colab import userdata
from pinecone import Pinecone, ServerlessSpec

PINECONE_API_KEY = userdata.get("PINECONE_API_KEY")
pc = Pinecone(api_key=PINECONE_API_KEY)
print("Cliente de Pinecone inicializado.")


Cliente de Pinecone inicializado.


## 4. Crear (o conectar a) el índice

Se crea un índice *serverless* — la modalidad que cubre el plan gratuito de Pinecone. La celda es segura de re-ejecutar: si el índice ya existe (por ejemplo, en una sesión anterior), no lo vuelve a crear, solo se conecta.

**Nota:** la combinación `cloud`/`region` disponible en el plan gratuito puede cambiar; si `us-east-1` en AWS da error de región no disponible, revisa en el dashboard de Pinecone qué región gratuita está habilitada para tu cuenta y ajusta el `ServerlessSpec`.

In [6]:
NOMBRE_INDICE = "ofertas-laborales"
DIMENSION = embeddings.shape[1]  # 384 para all-MiniLM-L6-v2

indices_existentes = [i["name"] for i in pc.list_indexes()]

if NOMBRE_INDICE not in indices_existentes:
    print(f"Creando índice '{NOMBRE_INDICE}'...")
    pc.create_index(
        name=NOMBRE_INDICE,
        dimension=DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    # Espera a que el índice quede listo antes de usarlo
    while not pc.describe_index(NOMBRE_INDICE).status["ready"]:
        time.sleep(1)
else:
    print(f"El índice '{NOMBRE_INDICE}' ya existe, se reutiliza.")

pinecone_index = pc.Index(NOMBRE_INDICE)
print(pinecone_index.describe_index_stats())


Creando índice 'ofertas-laborales'...
DescribeIndexStatsResponse(dimension=384, total_vector_count=0, metric='cosine', namespaces=0)


## 5. Subir (upsert) las plantillas al índice

Se sube cada plantilla como un vector con:
- **id**: el `template_id`, convertido a texto (Pinecone requiere IDs tipo string).
- **values**: el embedding de 384 dimensiones.
- **metadata**: `Job Title` y `Role`, para poder inspeccionar resultados directamente desde Pinecone sin tener que volver a cruzar con `df_clean` (aunque para el filtrado por país/salario/modalidad igual se usará `df_clean`, ver sección 7).

Se sube en lotes de 100 (límite recomendado por Pinecone por request).

In [7]:
vectores = [
    {
        "id": str(fila["template_id"]),
        "values": embeddings[i].tolist(),
        "metadata": {"Job Title": fila["Job Title"], "Role": fila["Role"]},
    }
    for i, fila in plantillas.iterrows()
]

TAMANO_LOTE = 100
for inicio_lote in range(0, len(vectores), TAMANO_LOTE):
    lote = vectores[inicio_lote:inicio_lote + TAMANO_LOTE]
    pinecone_index.upsert(vectors=lote)

print(f"{len(vectores):,} plantillas subidas a Pinecone.")
print(pinecone_index.describe_index_stats())


3,760 plantillas subidas a Pinecone.
DescribeIndexStatsResponse(dimension=384, total_vector_count=3760, metric='cosine', namespaces=1)


## 6. Cargar el modelo de embeddings (para codificar consultas)

Los embeddings de las plantillas ya están calculados y subidos — el modelo solo hace falta ahora para convertir **la consulta del usuario** en un vector, igual que en el notebook 03.

In [8]:
from sentence_transformers import SentenceTransformer

MODELO_EMBEDDINGS = "all-MiniLM-L6-v2"  # el elegido tras la comparación en 03_Embeddings_BusquedaVectorial.ipynb

modelo = SentenceTransformer(MODELO_EMBEDDINGS)
print(f"Modelo cargado: {MODELO_EMBEDDINGS}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo cargado: all-MiniLM-L6-v2


## 7. Búsqueda con Pinecone + expansión y filtros sobre `df_clean`

Misma lógica de `buscar_ofertas()` del notebook 03 (buscar plantillas relevantes → expandir a ofertas reales → filtrar por atributos estructurados), cambiando únicamente el motor de búsqueda vectorial: FAISS por Pinecone.

In [9]:
def buscar_ofertas_pinecone(consulta: str, k_plantillas: int = 5, max_resultados: int = 20, filtros: dict | None = None):
    vector_consulta = modelo.encode(consulta, normalize_embeddings=True, convert_to_numpy=True).tolist()

    respuesta = pinecone_index.query(
        vector=vector_consulta,
        top_k=k_plantillas,
        include_metadata=False,  # no hace falta: los metadatos definitivos salen de df_clean
    )

    template_ids_relevantes = [int(match["id"]) for match in respuesta["matches"]]
    similitud_por_template = {int(match["id"]): match["score"] for match in respuesta["matches"]}

    resultados = df_clean[df_clean["template_id"].isin(template_ids_relevantes)].copy()
    resultados["similitud"] = resultados["template_id"].map(similitud_por_template)

    if filtros:
        for columna, valor in filtros.items():
            resultados = resultados[resultados[columna] == valor]

    resultados = resultados.sort_values("similitud", ascending=False)
    return resultados[["Job Title", "Role", "Country", "Work Type", "Salary Range", "similitud"]].head(max_resultados)

# Ejemplo
buscar_ofertas_pinecone("python developer with machine learning and NLP experience", k_plantillas=5)


,Job Title,Role,Country,Work Type,Salary Range,similitud
1579429,Data Scientist,Machine Learning Engineer,Philippines,Part-Time,$63K-$108K,0.528813
1564320,Data Scientist,Machine Learning Engineer,Netherlands,Contract,$57K-$89K,0.528813
8866,Data Scientist,Machine Learning Engineer,Burkina Faso,Temporary,$56K-$115K,0.528813
1578792,Data Scientist,Machine Learning Engineer,South Africa,Contract,$55K-$118K,0.528813
1568256,Data Scientist,Machine Learning Engineer,Malawi,Full-Time,$59K-$88K,0.528813
1564434,Data Scientist,Machine Learning Engineer,Portugal,Intern,$64K-$90K,0.528813
734900,Data Scientist,Machine Learning Engineer,Somalia,Full-Time,$59K-$93K,0.528813
738197,Data Scientist,Machine Learning Engineer,"Bahamas, The",Temporary,$61K-$108K,0.528813
739743,Data Scientist,Machine Learning Engineer,Moldova,Intern,$57K-$98K,0.528813
710950,Data Scientist,Machine Learning Engineer,British Virgin Islands,Full-Time,$60K-$106K,0.528813


In [10]:
# Ejemplo con filtro estructurado
buscar_ofertas_pinecone(
    "python developer with machine learning and NLP experience",
    k_plantillas=5,
    filtros={"Work Type": "Full-Time"},
)


,Job Title,Role,Country,Work Type,Salary Range,similitud
1605706,Data Scientist,Machine Learning Engineer,Palau,Full-Time,$55K-$117K,0.528813
1555755,Data Scientist,Machine Learning Engineer,Andorra,Full-Time,$57K-$112K,0.528813
1459771,Data Scientist,Machine Learning Engineer,Guam,Full-Time,$55K-$97K,0.528813
1523747,Data Scientist,Machine Learning Engineer,Ghana,Full-Time,$57K-$87K,0.528813
1511162,Data Scientist,Machine Learning Engineer,Libya,Full-Time,$58K-$126K,0.528813
1529914,Data Scientist,Machine Learning Engineer,Kosovo,Full-Time,$63K-$112K,0.528813
1139789,Data Scientist,Machine Learning Engineer,Cambodia,Full-Time,$56K-$86K,0.528813
1548246,Data Scientist,Machine Learning Engineer,South Africa,Full-Time,$64K-$117K,0.528813
1548819,Data Scientist,Machine Learning Engineer,Sierra Leone,Full-Time,$61K-$81K,0.528813
405503,Data Scientist,Machine Learning Engineer,New Caledonia,Full-Time,$60K-$100K,0.528813
